# SynBioCrow 2.3 — Galaxy-SynBioCAD Literature Benchmark Ingest\nDownloads and inventories Galaxy-SynBioCAD Supplementary Dataset 2 (77 literature pathways / 60 expressed compounds). This run accesses benchmark truth for curation only; it does not generate or score SynBioCrow predictions.\n

In [ ]:
import os,sys,subprocess,shutil,json,hashlib,zipfile,requests
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive",force_remount=False)

REPO=Path("/content/SynBioCrow_galaxy_lit_ingest")
OUT=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_dataset2_ingest")
OUT.mkdir(parents=True,exist_ok=True)
XLSX=Path("/content/Galaxy_SynBioCAD_Supplementary_Dataset_2.xlsx")

def run(cmd,*,cwd=None,timeout=1200,check=True):
    print("$"," ".join(map(str,cmd)),flush=True)
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,
                     stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(p.stdout,flush=True)
    if check and p.returncode:
        raise subprocess.CalledProcessError(p.returncode,cmd,output=p.stdout)
    return p

if REPO.exists(): shutil.rmtree(REPO)
run(["git","clone","--depth","1","--branch","develop/2.3",
     "https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)

urls=[
 "https://media.springernature.com/original/springer-static/esm/art%3A10.1038%2Fs41467-022-32661-x/MediaObjects/41467_2022_32661_MOESM6_ESM.xlsx",
 "https://media.springernature.com/original/springer-static/esm/art:10.1038/s41467-022-32661-x/MediaObjects/41467_2022_32661_MOESM6_ESM.xlsx",
]
downloaded=False
for u in urls:
    try:
        print("[DATASET2] downloading",u,flush=True)
        r=requests.get(u,timeout=60)
        print("[DATASET2] status",r.status_code,"bytes",len(r.content),flush=True)
        if r.ok and len(r.content)>10000 and r.content[:2]==b"PK":
            XLSX.write_bytes(r.content)
            downloaded=True
            break
    except Exception as e:
        print("[DATASET2] download error",type(e).__name__,e,flush=True)

if not downloaded:
    print("\nAutomatic download failed. Upload the published Supplementary Dataset 2 XLSX.",flush=True)
    up=files.upload()
    candidates=[Path("/content")/name for name in up if name.lower().endswith(".xlsx")]
    assert candidates,"No XLSX uploaded"
    shutil.copy2(candidates[0],XLSX)

print("[DATASET2] file",XLSX,"bytes",XLSX.stat().st_size,
      "sha256",hashlib.sha256(XLSX.read_bytes()).hexdigest(),flush=True)

run([sys.executable,str(REPO/"scripts/inspect_galaxy_dataset2_2_3.py"),
     "--xlsx",str(XLSX),"--output-dir",str(OUT)],timeout=600)

bundle=OUT/"SynBioCrow_2_3_GALAXY_DATASET2_INGEST_RESULTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.glob("*.json")):
        z.write(p,p.name)
print("BUNDLE",bundle)
print("SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest())
files.download(str(bundle))
